# DuckDB Reporting Benchmarks

In [1]:
import duckdb
import time

# Connect to the DuckDB file
con = duckdb.connect('jaffle_shop.duckdb')
con.sql('SHOW TABLES')

┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ customers     │
│ orders        │
│ raw_customers │
│ raw_orders    │
│ raw_payments  │
│ stg_customers │
│ stg_orders    │
│ stg_payments  │
└───────────────┘

## Preview Tables

In [2]:
con.sql('SELECT * FROM customers LIMIT 5')

┌─────────────┬────────────┬───────────┬─────────────┬───────────────────┬──────────────────┬─────────────────────────┐
│ customer_id │ first_name │ last_name │ first_order │ most_recent_order │ number_of_orders │ customer_lifetime_value │
│    int32    │  varchar   │  varchar  │    date     │       date        │      int64       │         double          │
├─────────────┼────────────┼───────────┼─────────────┼───────────────────┼──────────────────┼─────────────────────────┤
│           1 │ Michael    │ P.        │ 2018-01-01  │ 2018-02-10        │                2 │                    33.0 │
│           2 │ Shawn      │ M.        │ 2018-01-11  │ 2018-01-11        │                1 │                    23.0 │
│           3 │ Kathleen   │ P.        │ 2018-01-02  │ 2018-03-11        │                3 │                    65.0 │
│           6 │ Sarah      │ R.        │ 2018-02-19  │ 2018-02-19        │                1 │                     8.0 │
│           7 │ Martin     │ M.        │

In [3]:
con.sql('SELECT * FROM orders LIMIT 5')

┌──────────┬─────────────┬────────────┬───┬───────────────┬──────────────────────┬──────────────────┬────────┐
│ order_id │ customer_id │ order_date │ … │ coupon_amount │ bank_transfer_amount │ gift_card_amount │ amount │
│  int32   │    int32    │    date    │   │    double     │        double        │      double      │ double │
├──────────┼─────────────┼────────────┼───┼───────────────┼──────────────────────┼──────────────────┼────────┤
│        1 │           1 │ 2018-01-01 │ … │           0.0 │                  0.0 │              0.0 │   10.0 │
│        2 │           3 │ 2018-01-02 │ … │           0.0 │                  0.0 │              0.0 │   20.0 │
│        3 │          94 │ 2018-01-04 │ … │           1.0 │                  0.0 │              0.0 │    1.0 │
│        4 │          50 │ 2018-01-05 │ … │          25.0 │                  0.0 │              0.0 │   25.0 │
│        5 │          64 │ 2018-01-05 │ … │           0.0 │                 17.0 │              0.0 │   17.0 │
├

In [4]:
con.sql('SELECT * FROM raw_payments LIMIT 5')

┌───────┬──────────┬────────────────┬────────┐
│  id   │ order_id │ payment_method │ amount │
│ int32 │  int32   │    varchar     │ int32  │
├───────┼──────────┼────────────────┼────────┤
│     1 │        1 │ credit_card    │   1000 │
│     2 │        2 │ credit_card    │   2000 │
│     3 │        3 │ coupon         │    100 │
│     4 │        4 │ coupon         │   2500 │
│     5 │        5 │ bank_transfer  │   1700 │
└───────┴──────────┴────────────────┴────────┘

## Benchmarking Helper

In [5]:
def run_and_time_query(query: str):
    start = time.time()
    result = con.execute(query).fetchall()
    end = time.time()
    runtime_ms = round((end - start) * 1000, 2)
    return result, runtime_ms

### 1. Top 10 customers

In [6]:
query1 = '''SELECT customer_id, SUM(amount) AS total_spent
FROM orders
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10'''

In [7]:
result1, time1 = run_and_time_query(query1)
print(time1)

0.92


### 2. Monthly revenue trend

In [8]:
query2 = '''SELECT DATE_TRUNC('month', order_date) AS month, SUM(orders.amount) AS revenue
FROM orders
JOIN raw_payments USING(order_id)
GROUP BY month
ORDER BY month'''

In [9]:
result2, time2 = run_and_time_query(query2)
# print(result2)
print(time2)

1.42


### 3. Avg order value per customer

In [10]:
query3 = '''SELECT customer_id, AVG(amount) AS avg_order_value
FROM orders
GROUP BY customer_id'''

In [11]:
result3, time3 = run_and_time_query(query3)
print(time3)

0.83


### 4. Number of orders per customer

In [12]:
query4 = '''SELECT customer_id, COUNT(order_id) AS num_orders
FROM orders
GROUP BY customer_id
ORDER BY num_orders DESC'''

In [13]:
result4, time4 = run_and_time_query(query4)
# result4
print(time4)

0.67


### 5. Revenue per order

In [14]:
query5 = '''SELECT order_id, SUM(amount) AS order_revenue
FROM raw_payments
GROUP BY order_id
ORDER BY order_revenue DESC'''

In [15]:
result5, time5 = run_and_time_query(query5)
# result5
print(time5)

0.82


### 6. Active customers per month

In [16]:
query6 = '''SELECT DATE_TRUNC('month', order_date) AS month, COUNT(DISTINCT customer_id) AS active_customers
FROM orders
GROUP BY month
ORDER BY month'''

In [17]:
result6, time6 = run_and_time_query(query6)
# result6
print(time6)

1.43


### 7. Average payment per order

In [18]:
query7 = '''SELECT order_id, AVG(amount) AS avg_payment
FROM raw_payments
GROUP BY order_id'''

In [19]:
result7, time7 = run_and_time_query(query7)
# result7
print(time7)

0.7


## Query Runtime Summary

In [20]:
runtimes = [
    ('Top 10 customers', time1),
    ('Monthly revenue trend', time2),
    ('Avg order value per customer', time3),
    ('Number of orders per customer', time4),
    ('Revenue per order', time5),
    ('Active customers per month', time6),
    ('Average payment per order', time7),
]

print('\nQuery Runtime Summary (ms):')
for name, ms in runtimes:
    print(f'{name:<35} : {ms} ms')


Query Runtime Summary (ms):
Top 10 customers                    : 0.92 ms
Monthly revenue trend               : 1.42 ms
Avg order value per customer        : 0.83 ms
Number of orders per customer       : 0.67 ms
Revenue per order                   : 0.82 ms
Active customers per month          : 1.43 ms
Average payment per order           : 0.7 ms


## Polars

In [21]:
import polars as pl
import os
db_path = os.path.abspath("../jaffle_shop.duckdb")
def run_query_to_polars(query: str):
    start = time.time()
    arrow_table = con.execute(query).arrow()
    df = pl.from_arrow(arrow_table)
    end = time.time()
    runtime_ms = round((end - start) * 1000, 2)
    return df, runtime_ms



In [22]:
polars_result1, polars_time1 = run_query_to_polars(query1)
polars_result2, polars_time2 = run_query_to_polars(query2)
polars_result3, polars_time3 = run_query_to_polars(query3)
polars_result4, polars_time4 = run_query_to_polars(query4)
polars_result5, polars_time5 = run_query_to_polars(query5)
polars_result6, polars_time6 = run_query_to_polars(query6)
polars_result7, polars_time7 = run_query_to_polars(query7)


In [23]:
runtimes = [
    ('Top 10 customers by LTV', polars_time1),
    ('Monthly revenue trend', polars_time2),
    ('Avg order value per customer', polars_time3),
    ('Number of orders per customer', polars_time4),
    ('Revenue per order', polars_time5),
    ('Active customers per month', polars_time6),
    ('Average payment per order', polars_time7),
]

print('\nQuery Runtime Summary (ms):')
for name, ms in runtimes:
    print(f'{name:<35} : {ms} ms')


Query Runtime Summary (ms):
Top 10 customers by LTV             : 86.06 ms
Monthly revenue trend               : 2.09 ms
Avg order value per customer        : 0.46 ms
Number of orders per customer       : 0.68 ms
Revenue per order                   : 0.89 ms
Active customers per month          : 1.1 ms
Average payment per order           : 0.44 ms


In [24]:
# polars_result1

# Comparison


In [28]:
benchmark_comparison = [
    ("Top 10 customers by LTV", time1, polars_time1),
    ("Monthly revenue trend", time2, polars_time2),
    ("Avg order value per customer", time3, polars_time3),
    ("Number of orders per customer", time4, polars_time4),
    ("Revenue per order", time5, polars_time5),
    ("Active customers per month", time6, polars_time6),
    ("Average payment per order", time7, polars_time7)
]
print(f"{'Query':<40} | {'DuckDB (ms)':>12} | {'Polars (ms)':>12} | {'Difference (ms)':>15}")
print("-" * 85)
for name, duck, polars in benchmark_comparison:
    diff = round(duck - polars, 2)
    print(f"{name:<40} | {duck:>12} | {polars:>12} | {diff:>15}")
    ## aligning the columns and choosing the right width


Query                                    |  DuckDB (ms) |  Polars (ms) | Difference (ms)
-------------------------------------------------------------------------------------
Top 10 customers by LTV                  |         0.92 |        86.06 |          -85.14
Monthly revenue trend                    |         1.42 |         2.09 |           -0.67
Avg order value per customer             |         0.83 |         0.46 |            0.37
Number of orders per customer            |         0.67 |         0.68 |           -0.01
Revenue per order                        |         0.82 |         0.89 |           -0.07
Active customers per month               |         1.43 |          1.1 |            0.33
Average payment per order                |          0.7 |         0.44 |            0.26
